<a href="https://colab.research.google.com/github/yse-eds-cert/code-along-notebooks/blob/main/live-sessions/module_8/key_M8_breakout_spot_the_bug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
# This is formatted as code
```

# ANSWER KEY

## Before you start, check runtime!

Google Colab defaults to a Python runtime. Because this notebook is written in R, you need to switch the runtime once before running any code:

1. In the Colab menu bar, click **Runtime → Change runtime type**.
2. Under **Runtime type**, select **R**.
3. Click **Save**. Colab will reload the notebook with R.

# Module 8 Breakout: Spot the Bug (Answer Key)

All three examples use today's running example: daily credit card spending at two stores, and the daily maximum temperature (`tmax_c`, °C) in each store's city.

Each example below runs **without an error**, but the result is wrong. In your group, spend about 5 minutes on each one and answer:

1. **What went wrong?** Point to the clue in the output.
2. **How could you have caught it?** What would you check?
3. **How would you fix the code?**

Each example has a hint you can click open. Try on your own for a couple of minutes first!

**For mentors.** Let groups struggle for a minute before hinting. Hints if a group is stuck: Example 1, "count the rows"; Example 2, "compare the two date columns"; Example 3, "look at the spend column". (Students can also click these open themselves.)

## Setup

Run the cell below to load the tidyverse and create the small tables we'll use. You don't need to read it closely: we'll print each table when we use it.

In [1]:
library(tidyverse)

# Daily credit card spending at two stores
spend_long <- tribble(
  ~store,           ~city,     ~date,        ~spend,
  "Coffee Shop",    "Phoenix", "2024-07-01",  412,
  "Coffee Shop",    "Phoenix", "2024-07-02",  380,
  "Coffee Shop",    "Phoenix", "2024-07-03",  295,
  "Hardware Store", "Atlanta", "2024-07-01", 1150,
  "Hardware Store", "Atlanta", "2024-07-02",  990,
  "Hardware Store", "Atlanta", "2024-07-03",  870
)

# Daily maximum temperature (°C) in each city
weather <- tribble(
  ~city,     ~date,        ~tmax_c,
  "Phoenix", "2024-07-01", 41,
  "Phoenix", "2024-07-02", 43,
  "Phoenix", "2024-07-03", 44,
  "Atlanta", "2024-07-01", 32,
  "Atlanta", "2024-07-02", 33,
  "Atlanta", "2024-07-03", 35
)

# Example 2: temperature data from a different source
weather_new <- tribble(
  ~city,     ~date,      ~tmax_c,
  "Phoenix", "7/1/2024", 41,
  "Phoenix", "7/2/2024", 43,
  "Phoenix", "7/3/2024", 44,
  "Atlanta", "7/1/2024", 32,
  "Atlanta", "7/2/2024", 33,
  "Atlanta", "7/3/2024", 35
)

# Example 3: a newer pull of the joined spending + temperature table
spend_temp_v2 <- tribble(
  ~store,           ~city,     ~date,        ~spend, ~tmax_c,
  "Coffee Shop",    "Phoenix", "2024-07-01",  412,   41,
  "Coffee Shop",    "Phoenix", "2024-07-02",  380,   43,
  "Coffee Shop",    "Phoenix", "2024-07-03",  295,   44,
  "Hardware Store", "Atlanta", "2024-07-01", 1150,   32,
  "Hardware Store", "Atlanta", "2024-07-02",  990,   33,
  "Hardware Store", "Atlanta", "2024-07-03",   NA,   35
)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Example 1: Adding temperature to spending

**Goal:** add each day's maximum temperature to each store's daily spending.

Here are the two tables we're joining.

In [2]:
spend_long

store,city,date,spend
<chr>,<chr>,<chr>,<dbl>
Coffee Shop,Phoenix,2024-07-01,412
Coffee Shop,Phoenix,2024-07-02,380
Coffee Shop,Phoenix,2024-07-03,295
Hardware Store,Atlanta,2024-07-01,1150
Hardware Store,Atlanta,2024-07-02,990
Hardware Store,Atlanta,2024-07-03,870


In [3]:
weather

city,date,tmax_c
<chr>,<chr>,<dbl>
Phoenix,2024-07-01,41
Phoenix,2024-07-02,43
Phoenix,2024-07-03,44
Atlanta,2024-07-01,32
Atlanta,2024-07-02,33
Atlanta,2024-07-03,35


Here's the code. Run it, then look closely at the result *and* anything else R prints.

In [4]:
spend_temp <- spend_long %>%
  left_join(weather, by = "date")

spend_temp

Warning message in left_join(., weather, by = "date"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


store,city.x,date,spend,city.y,tmax_c
<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>
Coffee Shop,Phoenix,2024-07-01,412,Phoenix,41
Coffee Shop,Phoenix,2024-07-01,412,Atlanta,32
Coffee Shop,Phoenix,2024-07-02,380,Phoenix,43
Coffee Shop,Phoenix,2024-07-02,380,Atlanta,33
Coffee Shop,Phoenix,2024-07-03,295,Phoenix,44
Coffee Shop,Phoenix,2024-07-03,295,Atlanta,35
Hardware Store,Atlanta,2024-07-01,1150,Phoenix,41
Hardware Store,Atlanta,2024-07-01,1150,Atlanta,32
Hardware Store,Atlanta,2024-07-02,990,Phoenix,43


### Your turn

<details>
<summary><b>Stuck? Click for a hint</b></summary>

Count the rows.
</details>

**1. What went wrong?**

*✏️ Write your answer here.*

ANSWER: **Joined on the wrong key.** The join used only `date`. The weather table has **two rows per date** (one per city), so every spending row matched twice. Six rows became twelve, and half of them are wrong: the Phoenix coffee shop is paired with Atlanta's temperature (row 2).

Clues in the output:
- 12 rows instead of 6.
- `city.x` and `city.y`: R adds `.x`/`.y` when both tables have a column with the same name that isn't a join key. If you see these suffixes, ask whether that column should have been a key.
- The many-to-many warning. Warnings are worth reading!

**2. What are some ways you could check the join to catch this error?** What would you check? Try your check in the code cell below if you like.

*✏️ Write your answer here.*

ANSWER:
- Compare `nrow()` before and after a `left_join()`. They should be equal: `nrow(spend_long); nrow(spend_temp)` gives 6 and 12.
- Read the warnings, and look for `.x`/`.y` suffixes in the column names.
- Ask: what makes one row unique in each table? For the weather table, it's `city` *and* `date`. You can check: `weather %>% count(date)` shows each date twice, while `weather %>% count(city, date) %>% filter(n > 1)` returns 0 rows.

**3. How would you fix the code?** Write the corrected code below and run it.

In [5]:
# ✏️ Your code here
spend_temp <- spend_long %>%
  left_join(weather, by = c("city", "date"))

spend_temp

# Check: same number of rows as spend_long (6)
nrow(spend_temp)

store,city,date,spend,tmax_c
<chr>,<chr>,<chr>,<dbl>,<dbl>
Coffee Shop,Phoenix,2024-07-01,412,41
Coffee Shop,Phoenix,2024-07-02,380,43
Coffee Shop,Phoenix,2024-07-03,295,44
Hardware Store,Atlanta,2024-07-01,1150,32
Hardware Store,Atlanta,2024-07-02,990,33
Hardware Store,Atlanta,2024-07-03,870,35


[1] 6

## Example 2: Add weather to spending (still), but with a different weather file

**Goal:** merge the new weather data file, `weather_new` to your original spending data, `spend_long`. From Example 1, we know to join on both `city` and `date`. `spend_long` is the same as before.

Here's the new weather table.

In [6]:
weather_new

city,date,tmax_c
<chr>,<chr>,<dbl>
Phoenix,7/1/2024,41
Phoenix,7/2/2024,43
Phoenix,7/3/2024,44
Atlanta,7/1/2024,32
Atlanta,7/2/2024,33
Atlanta,7/3/2024,35


Here is the code you'll try this time. There is a different error. Run it and look at the result. Try to identify what is going on with the join, now.


In [7]:
spend_temp <- spend_long %>%
  left_join(weather_new, by = c("city", "date"))

spend_temp

store,city,date,spend,tmax_c
<chr>,<chr>,<chr>,<dbl>,<dbl>
Coffee Shop,Phoenix,2024-07-01,412,NA
Coffee Shop,Phoenix,2024-07-02,380,NA
Coffee Shop,Phoenix,2024-07-03,295,NA
Hardware Store,Atlanta,2024-07-01,1150,NA
Hardware Store,Atlanta,2024-07-02,990,NA
Hardware Store,Atlanta,2024-07-03,870,NA


### Your turn

<details>
<summary><b>Stuck? Click for a hint</b></summary>

Compare the two `date` columns.
</details>

**1. What went wrong?**

*✏️ Write your answer here.*

ANSWER: **Date formats don't match.** The keys look like the same thing but aren't: `spend_long` has `"2024-07-01"` and the new weather file has `"7/1/2024"`. Both are stored as text (`<chr>`), so R compares them as strings, nothing matches, and there is no error. `left_join()` keeps every spending row and fills `tmax_c` with `NA`.

Clues in the output:
- Every value of `tmax_c` is `NA`.
- The `date` columns are `<chr>`, not `<date>`.

This is the same issue as "St. Louis" vs. "Saint Louis" from the slides: the values mean the same thing to a person but not to R.

**2. How would you fix the code?** Just describe the answer.

*✏️ Write your answer here.*

ANSWER: Turn both `date` columns into real dates (the `<date>` type) before joining, each with the lubridate function that matches its format: `ymd()` for `"2024-07-01"` and `mdy()` for `"7/1/2024"`. Then the join on `city` and `date` matches all 6 rows. (lubridate loads with the tidyverse.)

```r
spend_long_dates  <- spend_long  %>% mutate(date = ymd(date))   # "2024-07-01"
weather_new_dates <- weather_new %>% mutate(date = mdy(date))   # "7/1/2024"

spend_temp <- spend_long_dates %>%
  left_join(weather_new_dates, by = c("city", "date"))
```

If a group finishes early, ask how they could have caught it:
- `glimpse()` both tables before joining and compare the key columns: same type, same format?
- `anti_join(spend_long, weather_new, by = c("city", "date"))` returns all 6 rows: nothing matched.
- Count missing values after a join: `sum(is.na(spend_temp$tmax_c))`.

## Example 3: Average spending per store

**Goal:** find each store's average daily spending. For this one, we're using `spend_temp_v2`, a newer version of the joined table.

In [8]:
spend_temp_v2

store,city,date,spend,tmax_c
<chr>,<chr>,<chr>,<dbl>,<dbl>
Coffee Shop,Phoenix,2024-07-01,412,41
Coffee Shop,Phoenix,2024-07-02,380,43
Coffee Shop,Phoenix,2024-07-03,295,44
Hardware Store,Atlanta,2024-07-01,1150,32
Hardware Store,Atlanta,2024-07-02,990,33
Hardware Store,Atlanta,2024-07-03,NA,35


Here's the code you'll try first. Run it and look at the result. There is a problem. Try to figure out what is going wrong, and how you can fix it.

In [9]:
spend_temp_v2 %>%
  group_by(store) %>%
  summarize(mean_spend = mean(spend))

store,mean_spend
<chr>,<dbl>
Coffee Shop,362.3333
Hardware Store,NA


### Your turn

<details>
<summary><b>Stuck? Click for a hint</b></summary>

Look at the `spend` column.
</details>

**1. What went wrong?**    

*✏️ Write your answer here.*

ANSWER: **A missing value.** The Hardware Store is missing spending on 2024-07-03. `mean()` returns `NA` if *any* value is `NA`, so that store's whole average is `NA`.

Clues:
- `mean_spend` is `NA` for one group only.
- The input has an `NA` in `spend` (row 6).

**2. How could you have caught it?** What would you check? Try your check in the code cell below if you like.

*✏️ Write your answer here.*

ANSWER:
- Check for missing values before summarizing: `summary()`, or `sum(is.na(spend))`.
- Report the number of missing values in the same `summarize()` (see the fix below), so you always see it.

In [10]:
# ✏️ Your code here
# summary() reports NA's for each numeric column
summary(spend_temp_v2)

sum(is.na(spend_temp_v2$spend))

       store           city          date        spend            tmax_c    
 Length   : 6   Length   :6   Length   : 6   Min.   : 295.0   Min.   :32.0  
 N.unique : 2   N.unique :2   N.unique : 3   1st Qu.: 380.0   1st Qu.:33.5  
 N.blank  : 0   N.blank  :0   N.blank  : 0   Median : 412.0   Median :38.0  
 Min.nchar:11   Min.nchar:7   Min.nchar:10   Mean   : 645.4   Mean   :38.0  
 Max.nchar:14   Max.nchar:7   Max.nchar:10   3rd Qu.: 990.0   3rd Qu.:42.5  
                                             Max.   :1150.0   Max.   :44.0  
                                             NAs    :1                      

[1] 1

**3. How would you fix the code?** Describe your fix and try to implement it.

✏️ Your answer here

ANSWER: Add `na.rm = TRUE` so `mean()` skips the missing value. It's good practice to also count the missing values in the same `summarize()`, so you can see how many were dropped.

```r
spend_temp_v2 %>%
  group_by(store) %>%
  summarize(mean_spend = mean(spend, na.rm = TRUE),
            n_missing  = sum(is.na(spend)))
```

This gives 362 for the Coffee Shop and 1070 for the Hardware Store, with `n_missing` of 0 and 1.